# §4.3 — Spatial Patterns & Hotspots

KDE hotspot map, Ripley's K, NND histogram, Local Moran's I on fractal dimension.

**Inputs:** `data/clusters_typed.csv`  
**Outputs:** Figures 7a, 7b, 7c, 8

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from metrics.spatial import nnd_stats, ripley_k, local_morans_i

In [ ]:
df = pd.read_csv('../data/clusters_typed.csv').dropna(subset=['cx', 'cy'])
coords = df[['cx', 'cy']].values
print(f'{len(df):,} clusters')

## Figure 7a — Adaptive KDE hotspot map

In [ ]:
from scipy.stats import gaussian_kde

kde = gaussian_kde(coords.T, bw_method='scott')
xi = np.linspace(coords[:, 0].min(), coords[:, 0].max(), 200)
yi = np.linspace(coords[:, 1].min(), coords[:, 1].max(), 200)
Xi, Yi = np.meshgrid(xi, yi)
Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)

fig, ax = plt.subplots(figsize=(9, 8), dpi=150)
cf = ax.contourf(Xi, Yi, Zi, levels=20, cmap='viridis', alpha=0.8)
ax.scatter(coords[:, 0], coords[:, 1], s=2, color='red', alpha=0.2)
plt.colorbar(cf, ax=ax, label='Kernel density')
ax.set_title('(a) Adaptive KDE hotspot map', fontsize=12)
ax.set_xlabel('Easting (m)'); ax.set_ylabel('Northing (m)')
plt.tight_layout()
plt.savefig('../figures/fig7a_kde.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 7b — Ripley's K function

Assigns 125 spatial plot IDs via K-means, then computes K per plot (parallelised).

In [ ]:
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)
plot_ids = KMeans(n_clusters=125, random_state=42, n_init=5).fit_predict(coords_scaled)

d_vals, k_obs = ripley_k(coords, plot_ids, max_dist=1500, n_intervals=50,
                         plot_size=6000, n_jobs=-1)
k_csr = np.pi * d_vals ** 2

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
ax.plot(d_vals, k_obs, color='#0072B2', lw=2, label='Observed K (mean)')
ax.plot(d_vals, k_csr, '--', color='#D55E00', lw=2, label='Theoretical CSR')
ax.set_xlabel('Distance (m)', fontsize=12)
ax.set_ylabel('K(d)', fontsize=12)
ax.set_title("(b) Ripley's K function", fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../figures/fig7b_ripley_k.png', dpi=300, bbox_inches='tight')
plt.show()

for d in [50, 100, 200]:
    idx = np.argmin(np.abs(d_vals - d))
    print(f'd={d} m: obs={k_obs[idx]:.0f}  CSR={k_csr[idx]:.0f}')

## Figure 7c — NND histogram

In [ ]:
from metrics.spatial import nnd_array
nnd = nnd_array(coords)
area_total = (coords[:, 0].ptp()) * (coords[:, 1].ptp())
lambda_hat = len(coords) / area_total
expected_nnd = 0.5 / np.sqrt(lambda_hat)

print(f'Mean NND: {nnd.mean():.1f} m  (expect ~30 m)')
print(f'Expected NND under CSR: {expected_nnd:.1f} m  (expect ~45 m)')

fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
ax.hist(nnd, bins=60, color='#4477AA', edgecolor='white', lw=0.3, density=True, alpha=0.8)
ax.axvline(nnd.mean(), color='red', lw=2, label=f'Observed mean = {nnd.mean():.1f} m')
ax.axvline(expected_nnd, color='orange', lw=2, ls='--',
           label=f'CSR expected = {expected_nnd:.1f} m')
ax.set_xlabel('NND (m)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('(c) Nearest-Neighbour Distance histogram', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../figures/fig7c_nnd.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 8 — Local Moran's I hotspots on fractal dimension

In [ ]:
sub = df.dropna(subset=['fractal_dim'])
lm = local_morans_i(sub[['cx', 'cy']].values, sub['fractal_dim'].values, threshold=1000.0)
sub = sub.copy()
sub['lm_label'] = lm['label'].values

label_colour = {'HH': '#D7191C', 'LL': '#2C7BB6', 'HL': '#FDAE61', 'LH': '#ABD9E9', 'NS': '#CCCCCC'}
fig, ax = plt.subplots(figsize=(10, 9), dpi=150)
for lbl, col in label_colour.items():
    pts = sub[sub.lm_label == lbl]
    if len(pts):
        ax.scatter(pts['cx'], pts['cy'], s=5, color=col, alpha=0.7, label=lbl)

ax.set_xlabel('Easting (m)', fontsize=11)
ax.set_ylabel('Northing (m)', fontsize=11)
ax.set_title('Local Moran\'s I — fractal dimension hotspots/coldspots', fontsize=12)
ax.legend(title='LISA label', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/fig8_local_morans.png', dpi=300, bbox_inches='tight')
plt.show()